# 미사용/미재사용 원인 분석 전용 노트북

- 목적: **미사용/미재사용(0~2건)** 과 **재사용(3건 이상)** 의 차이를 가설 기반으로 검증
- 핵심: `unanswered_count(답변불가경험건수)`를 실제로 분석에 포함
- 출력: 표는 `print_md_table()`로 마크다운 형식 출력 (보고서 복붙용)
- 시각화: 최소(필요한 1~2개만)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

def safe_div(n, d):
    if d is None or d == 0 or pd.isna(d):
        return np.nan
    return n / d

def print_md_table(title, df, digits=4):
    print("\n" + "=" * 95)
    print(title)
    print("=" * 95)
    if df is None or len(df) == 0:
        print("(데이터 없음)")
        return
    out = df.copy()
    num_cols = out.select_dtypes(include=["number"]).columns
    for c in num_cols:
        out[c] = out[c].round(digits)
    try:
        print(out.to_markdown(index=False))
    except Exception:
        print(out.to_string(index=False))

def first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

In [ ]:
# ===== 경로 / 기준값 설정 =====
DATA_DIR = Path("data")
OPEN_DATE = pd.Timestamp("2026-03-23")

profile_file = DATA_DIR / "customer_profile.csv"
chat_file = DATA_DIR / "ai_chat_daily_by_user.csv"
ai_transfer_file = DATA_DIR / "ai_transfer_daily_by_user.csv"

# 미재사용 기준: 총 AI 요청건수 <= 2건
NON_REUSE_MAX_REQ = 2
# 재사용 기준: 총 AI 요청건수 >= 3건
REUSE_MIN_REQ = 3

In [ ]:
def load_csv(path):
    if path.exists():
        return pd.read_csv(path)
    print(f"[WARN] 파일 없음: {path}")
    return pd.DataFrame()

def normalize_profile(df):
    if df.empty:
        return df
    mapper = {
        "customer_id": ["customer_id", "고객번호"],
        "age_band": ["age_band", "나이대"],
        "is_employee": ["is_employee", "임직원여부"],
        "loan_customer": ["loan_customer", "여신고객여부", "is_loan_customer", "loan_yn"],
        "loan_account_count": ["loan_account_count", "대출계좌건수", "loan_acct_cnt", "loan_cnt"],
        "ebank_signup_date": ["ebank_signup_date", "전자금융가입일"],
        "ai_signup_date": ["ai_signup_date", "AI가입일"],
        "last_login_date": ["last_login_date", "최근접속일"],
        "pre_year_avg_transfer_count": ["pre_year_avg_transfer_count", "AI가입전1년_월평균이체건수"],
        "pre30_transfer_count": ["pre30_transfer_count", "AI가입전30일_이체건수"],
        "post30_ai_transfer_count": ["post30_ai_transfer_count", "AI가입후30일_ai이체건수"],
        "post30_other_transfer_count": ["post30_other_transfer_count", "AI가입후30일_기존이체건수"],
        "stt_request_days": ["stt_request_days", "STT요청일수", "stt_days"],
        "stt_request_count": ["stt_request_count", "STT요청건수", "stt_count"],
        "menu_reco_request_count": ["menu_reco_request_count", "메뉴추천요청건수", "menu_request_count"],
        "feedback_like_count": ["feedback_like_count", "피드백좋아요건"],
        "feedback_dislike_count": ["feedback_dislike_count", "피드백싫어요건"],
        "unanswered_count": ["unanswered_count", "답변불가경험건수"],
    }

    rename = {}
    for std, cands in mapper.items():
        found = first_existing_column(df, cands)
        if found:
            rename[found] = std

    out = df.rename(columns=rename).copy()

    for c in ["ebank_signup_date", "ai_signup_date", "last_login_date"]:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c], errors="coerce")

    num_cols = [
        "pre_year_avg_transfer_count", "pre30_transfer_count", "post30_ai_transfer_count",
        "post30_other_transfer_count", "stt_request_days", "stt_request_count",
        "menu_reco_request_count", "feedback_like_count", "feedback_dislike_count", "unanswered_count", "loan_account_count"
    ]
    for c in num_cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    # loan_customer_flag 파생: (loan_customer가 있거나) 대출계좌건수>=1 이면 True
    loan_flag = pd.Series(False, index=out.index)

    if "loan_customer" in out.columns:
        s = out["loan_customer"].astype(str).str.strip().str.lower()
        loan_flag = loan_flag | s.isin(["1", "y", "yes", "true", "t", "여", "여신", "대출", "loan"])
        loan_flag = loan_flag | pd.to_numeric(out["loan_customer"], errors="coerce").fillna(0).gt(0)

    if "loan_account_count" in out.columns:
        loan_flag = loan_flag | pd.to_numeric(out["loan_account_count"], errors="coerce").fillna(0).ge(1)

    out["loan_customer_flag"] = loan_flag

    return out

def normalize_chat(df):
    if df.empty:
        return df
    mapper = {
        "customer_id": ["customer_id", "고객번호"],
        "ai_signup_date": ["ai_signup_date", "AI가입일"],
        "chat_date": ["chat_date", "채팅요청일", "요청일", "date"],
        "service_category": ["service_category", "서비스분류", "서비스분류코드"],
        "request_count": ["request_count", "건수", "cnt"],
    }

    rename = {}
    for std, cands in mapper.items():
        found = first_existing_column(df, cands)
        if found:
            rename[found] = std

    out = df.rename(columns=rename).copy()

    for c in ["ai_signup_date", "chat_date"]:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c], errors="coerce")

    if "request_count" not in out.columns:
        out["request_count"] = 1
    out["request_count"] = pd.to_numeric(out["request_count"], errors="coerce").fillna(0)

    return out

def normalize_ai_transfer(df):
    if df.empty:
        return df
    mapper = {
        "customer_id": ["customer_id", "고객번호"],
        "ai_signup_date": ["ai_signup_date", "AI가입일"],
        "ai_transfer_date": ["ai_transfer_date", "ai이체일"],
        "ai_transfer_count": ["ai_transfer_count", "ai이체건수"],
        "ai_transfer_amount": ["ai_transfer_amount", "금액합"],
    }

    rename = {}
    for std, cands in mapper.items():
        found = first_existing_column(df, cands)
        if found:
            rename[found] = std

    out = df.rename(columns=rename).copy()

    for c in ["ai_signup_date", "ai_transfer_date"]:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c], errors="coerce")

    for c in ["ai_transfer_count", "ai_transfer_amount"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0)
        else:
            out[c] = 0

    return out

In [ ]:
profile = normalize_profile(load_csv(profile_file))
chat = normalize_chat(load_csv(chat_file))
ai_transfer = normalize_ai_transfer(load_csv(ai_transfer_file))

print_md_table("데이터 로딩 현황", pd.DataFrame([
    {"dataset": "profile", "rows": len(profile), "columns": len(profile.columns)},
    {"dataset": "chat", "rows": len(chat), "columns": len(chat.columns)},
    {"dataset": "ai_transfer", "rows": len(ai_transfer), "columns": len(ai_transfer.columns)},
]))

In [ ]:
def build_user_base(profile_df, chat_df, ai_transfer_df):
    if "customer_id" not in profile_df.columns:
        raise ValueError("profile 데이터에 고객번호(customer_id)가 필요합니다.")

    base = profile_df.copy()
    base = base.drop_duplicates(subset=["customer_id"]).copy()

    # 채팅기반 사용자 집계
    if not chat_df.empty and "customer_id" in chat_df.columns:
        c = chat_df.copy()
        c = c[c["request_count"] > 0]

        user_chat = c.groupby("customer_id", as_index=False).agg(
            total_ai_requests=("request_count", "sum"),
            active_days=("chat_date", "nunique"),
            first_chat_date=("chat_date", "min"),
            last_chat_date=("chat_date", "max"),
        )

        # 서비스별 프록시 지표
        if "service_category" in c.columns:
            s = c.copy()
            s["service_category"] = s["service_category"].astype(str)

            s["is_stt_proxy"] = s["service_category"].str.contains("stt|음성|voice", case=False, na=False)
            s["is_menu_proxy"] = s["service_category"].str.contains("메뉴|menu|추천", case=False, na=False)
            s["is_transfer_proxy"] = s["service_category"].str.contains("이체|transfer", case=False, na=False)
            s["is_unanswered_proxy"] = s["service_category"].str.contains("답변불가|unanswered|not_answerable", case=False, na=False)

            proxy = s.groupby("customer_id", as_index=False).agg(
                stt_proxy_requests=("request_count", lambda x: x[s.loc[x.index, "is_stt_proxy"]].sum()),
                menu_proxy_requests=("request_count", lambda x: x[s.loc[x.index, "is_menu_proxy"]].sum()),
                transfer_proxy_requests=("request_count", lambda x: x[s.loc[x.index, "is_transfer_proxy"]].sum()),
                unanswered_proxy_requests=("request_count", lambda x: x[s.loc[x.index, "is_unanswered_proxy"]].sum()),
            )
            user_chat = user_chat.merge(proxy, on="customer_id", how="left")
        else:
            for col in ["stt_proxy_requests", "menu_proxy_requests", "transfer_proxy_requests", "unanswered_proxy_requests"]:
                user_chat[col] = np.nan
    else:
        user_chat = pd.DataFrame(columns=["customer_id", "total_ai_requests", "active_days", "first_chat_date", "last_chat_date"])

    base = base.merge(user_chat, on="customer_id", how="left")

    # AI이체 실행 집계
    if not ai_transfer_df.empty and "customer_id" in ai_transfer_df.columns:
        t = ai_transfer_df.groupby("customer_id", as_index=False).agg(
            ai_transfer_exec_count=("ai_transfer_count", "sum"),
            ai_transfer_exec_amount=("ai_transfer_amount", "sum"),
            ai_transfer_exec_days=("ai_transfer_date", "nunique"),
        )
        base = base.merge(t, on="customer_id", how="left")

    # 결측/타입 정리
    fill_zero_cols = [
        "total_ai_requests", "active_days", "stt_proxy_requests", "menu_proxy_requests",
        "transfer_proxy_requests", "unanswered_proxy_requests", "ai_transfer_exec_count",
        "ai_transfer_exec_amount", "ai_transfer_exec_days", "unanswered_count"
    ]
    for c in fill_zero_cols:
        if c in base.columns:
            base[c] = pd.to_numeric(base[c], errors="coerce").fillna(0)

    # STT / 메뉴 / 이체 지표(프로필 컬럼 우선, 없으면 프록시)
    if "stt_request_count" not in base.columns:
        base["stt_request_count"] = np.nan
    base["stt_signal_count"] = np.where(base["stt_request_count"].notna(), base["stt_request_count"], base.get("stt_proxy_requests", 0))

    if "menu_reco_request_count" not in base.columns:
        base["menu_reco_request_count"] = np.nan
    base["menu_signal_count"] = np.where(base["menu_reco_request_count"].notna(), base["menu_reco_request_count"], base.get("menu_proxy_requests", 0))

    if "post30_ai_transfer_count" not in base.columns:
        base["post30_ai_transfer_count"] = np.nan
    base["transfer_signal_count"] = np.where(base["post30_ai_transfer_count"].notna(), base["post30_ai_transfer_count"], base.get("ai_transfer_exec_count", 0))

    # 비율형 파생
    base["unanswered_rate"] = np.where(base["total_ai_requests"] > 0, base["unanswered_count"] / base["total_ai_requests"], np.nan)
    base["stt_share"] = np.where(base["total_ai_requests"] > 0, base["stt_signal_count"] / base["total_ai_requests"], np.nan)
    base["menu_share"] = np.where(base["total_ai_requests"] > 0, base["menu_signal_count"] / base["total_ai_requests"], np.nan)
    base["transfer_share"] = np.where(base["total_ai_requests"] > 0, base["transfer_signal_count"] / base["total_ai_requests"], np.nan)

    # 그룹 라벨
    base["reuse_group"] = np.select(
        [
            base["total_ai_requests"] <= NON_REUSE_MAX_REQ,
            base["total_ai_requests"] >= REUSE_MIN_REQ,
        ],
        ["미사용/미재사용(0~2건)", "재사용(3건 이상)"],
        default="중간구간",
    )

    # 분석대상: 0~2건 vs 3건 이상만 사용
    analysis = base[base["reuse_group"].isin(["미사용/미재사용(0~2건)", "재사용(3건 이상)"])].copy()

    return base, analysis

user_base, analysis_df = build_user_base(profile, chat, ai_transfer)

print_md_table("그룹 분포", analysis_df.groupby("reuse_group", as_index=False).agg(고객수=("customer_id", "nunique")))

In [ ]:
# 나이대 숫자화 (예: 20대 -> 20, 60세이상 -> 60)
def parse_age_band_to_num(x):
    if pd.isna(x):
        return np.nan
    s = str(x)
    nums = re.findall(r"\d+", s)
    if not nums:
        return np.nan
    return float(nums[0])

if "age_band" in analysis_df.columns:
    analysis_df["age_num"] = analysis_df["age_band"].apply(parse_age_band_to_num)
    analysis_df["is_senior_50p"] = analysis_df["age_num"] >= 50
else:
    analysis_df["age_num"] = np.nan
    analysis_df["is_senior_50p"] = np.nan

In [ ]:
# 통계 보조 함수 (scipy 없이 permutation 기반)
def permutation_pvalue(a, b, n_perm=1000, random_state=42):
    a = pd.Series(a).dropna().values
    b = pd.Series(b).dropna().values
    if len(a) < 2 or len(b) < 2:
        return np.nan

    rng = np.random.default_rng(random_state)
    observed = abs(a.mean() - b.mean())
    combined = np.concatenate([a, b])
    n_a = len(a)

    cnt = 0
    for _ in range(n_perm):
        rng.shuffle(combined)
        a_perm = combined[:n_a]
        b_perm = combined[n_a:]
        diff = abs(a_perm.mean() - b_perm.mean())
        if diff >= observed:
            cnt += 1
    return (cnt + 1) / (n_perm + 1)

def compare_numeric(df, feature, group_col="reuse_group", g1="재사용(3건 이상)", g0="미사용/미재사용(0~2건)"):
    if feature not in df.columns:
        return None
    s1 = pd.to_numeric(df.loc[df[group_col] == g1, feature], errors="coerce")
    s0 = pd.to_numeric(df.loc[df[group_col] == g0, feature], errors="coerce")
    m1, m0 = s1.mean(), s0.mean()
    p = permutation_pvalue(s1, s0)
    return {
        "feature": feature,
        "reuse_mean": m1,
        "nonreuse_mean": m0,
        "diff(reuse-nonreuse)": m1 - m0,
        "uplift_ratio(reuse/nonreuse)": safe_div(m1, m0),
        "perm_pvalue": p,
        "n_reuse": s1.notna().sum(),
        "n_nonreuse": s0.notna().sum(),
    }

def compare_binary_rate(df, feature, threshold=0, group_col="reuse_group", g1="재사용(3건 이상)", g0="미사용/미재사용(0~2건)"):
    if feature not in df.columns:
        return None
    x = pd.to_numeric(df[feature], errors="coerce")
    flag = x > threshold
    d = df.copy()
    d["_flag"] = flag
    s = d.groupby(group_col)["_flag"].mean()
    c = d.groupby(group_col)["_flag"].sum()
    n = d.groupby(group_col)["_flag"].count()

    reuse = s.get(g1, np.nan)
    nonreuse = s.get(g0, np.nan)

    # binary에도 mean 차이 permutation 사용
    p = permutation_pvalue(d.loc[d[group_col] == g1, "_flag"].astype(float), d.loc[d[group_col] == g0, "_flag"].astype(float))

    return {
        "feature": feature,
        "condition": f"> {threshold}",
        "reuse_rate": reuse,
        "nonreuse_rate": nonreuse,
        "diff(reuse-nonreuse)": reuse - nonreuse,
        "uplift_ratio(reuse/nonreuse)": safe_div(reuse, nonreuse),
        "perm_pvalue": p,
        "reuse_success": c.get(g1, np.nan),
        "reuse_n": n.get(g1, np.nan),
        "nonreuse_success": c.get(g0, np.nan),
        "nonreuse_n": n.get(g0, np.nan),
    }

## 가설 검증 (재사용자 특징)

가설 R1) 재사용자는 STT가 편해서 더 많이 쓸 것이다.  
가설 R2) 재사용자는 노년층 비중이 높을 것이다.  
가설 R3) 재사용자는 메뉴이동/메뉴추천 사용이 많을 것이다.  
가설 R4) 재사용자는 이체 활용이 많을 것이다.

In [ ]:
reuse_hyp_rows = []

# R1: STT
r1_num = compare_numeric(analysis_df, "stt_signal_count")
r1_rate = compare_binary_rate(analysis_df, "stt_signal_count", threshold=0)
if r1_num: reuse_hyp_rows.append({"가설": "R1_STT사용량", **r1_num})
if r1_rate: reuse_hyp_rows.append({"가설": "R1_STT경험률", **r1_rate})

# R2: 노년층
if "is_senior_50p" in analysis_df.columns:
    tmp = analysis_df.copy()
    tmp["is_senior_50p_num"] = tmp["is_senior_50p"].astype(float)
    r2 = compare_numeric(tmp, "is_senior_50p_num")
    if r2:
        reuse_hyp_rows.append({"가설": "R2_노년층비중(50+) ", **r2})

# R3: 메뉴추천/메뉴이동
r3_num = compare_numeric(analysis_df, "menu_signal_count")
r3_rate = compare_binary_rate(analysis_df, "menu_signal_count", threshold=0)
if r3_num: reuse_hyp_rows.append({"가설": "R3_메뉴사용량", **r3_num})
if r3_rate: reuse_hyp_rows.append({"가설": "R3_메뉴경험률", **r3_rate})

# R4: 이체활용
r4_num = compare_numeric(analysis_df, "transfer_signal_count")
r4_rate = compare_binary_rate(analysis_df, "transfer_signal_count", threshold=0)
if r4_num: reuse_hyp_rows.append({"가설": "R4_이체활용량", **r4_num})
if r4_rate: reuse_hyp_rows.append({"가설": "R4_이체경험률", **r4_rate})

reuse_hyp_df = pd.DataFrame(reuse_hyp_rows)
print_md_table("재사용자 가설 검증 결과", reuse_hyp_df)

## 가설 검증 (미사용/미재사용자 특징)

가설 N1) 미사용/미재사용자는 답변불가 경험이 많을 것이다.  
가설 N2) 미사용/미재사용자는 여신고객 비중이 높을 것이다.

In [ ]:
nonreuse_hyp_rows = []

# N1: 답변불가 경험 (핵심)
n1_num = compare_numeric(analysis_df, "unanswered_count")
n1_rate = compare_binary_rate(analysis_df, "unanswered_count", threshold=0)
n1_ratio = compare_numeric(analysis_df, "unanswered_rate")

if n1_num: nonreuse_hyp_rows.append({"가설": "N1_답변불가경험건수", **n1_num})
if n1_rate: nonreuse_hyp_rows.append({"가설": "N1_답변불가경험률", **n1_rate})
if n1_ratio: nonreuse_hyp_rows.append({"가설": "N1_답변불가비율", **n1_ratio})

# N2: 여신고객 (대출계좌건수>=1 포함)
if "loan_customer_flag" in analysis_df.columns:
    temp = analysis_df.copy()
    temp["loan_customer_num"] = temp["loan_customer_flag"].astype(float)
    n2 = compare_numeric(temp, "loan_customer_num")
    if n2:
        nonreuse_hyp_rows.append({"가설": "N2_여신고객비중", **n2})
else:
    print("[INFO] loan_customer_flag 생성 실패로 N2 가설은 스킵됩니다. (loan_customer 또는 대출계좌건수 컬럼 확인)")

nonreuse_hyp_df = pd.DataFrame(nonreuse_hyp_rows)
print_md_table("미사용/미재사용 가설 검증 결과", nonreuse_hyp_df)

## 추가 탐색: 데이터 기반 가설 후보 자동 발굴

- 사용 가능한 수치 변수 중에서 재사용군/미재사용군 차이가 큰 항목을 자동 추출
- 새 가설 수립 후보로 사용

In [ ]:
exclude_cols = {
    "customer_id", "total_ai_requests", "active_days", "reuse_group"
}
num_cols = [c for c in analysis_df.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(analysis_df[c])]

rows = []
for c in num_cols:
    comp = compare_numeric(analysis_df, c)
    if comp:
        rows.append(comp)

explore_df = pd.DataFrame(rows)
if not explore_df.empty:
    explore_df = explore_df.sort_values("diff(reuse-nonreuse)", ascending=False)

print_md_table("추가 가설 후보 TOP(+): 재사용군이 높은 항목", explore_df.head(10))
print_md_table("추가 가설 후보 TOP(-): 미재사용군이 높은 항목", explore_df.tail(10).sort_values("diff(reuse-nonreuse)"))

In [ ]:
# 최소 시각화: 핵심 가설 지표 비교
plot_features = [
    ("unanswered_count", "답변불가경험건수"),
    ("stt_signal_count", "STT 신호 건수"),
    ("menu_signal_count", "메뉴 신호 건수"),
    ("transfer_signal_count", "이체 신호 건수"),
]

plot_rows = []
for col, label in plot_features:
    if col in analysis_df.columns:
        g = analysis_df.groupby("reuse_group")[col].mean()
        plot_rows.append({"지표": label, "그룹": "재사용(3건 이상)", "평균": g.get("재사용(3건 이상)", np.nan)})
        plot_rows.append({"지표": label, "그룹": "미사용/미재사용(0~2건)", "평균": g.get("미사용/미재사용(0~2건)", np.nan)})

plot_df = pd.DataFrame(plot_rows)
if not plot_df.empty:
    plt.figure(figsize=(9, 4.5))
    sns.barplot(data=plot_df, x="지표", y="평균", hue="그룹")
    plt.title("핵심 가설 지표 평균 비교")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

In [ ]:
def verdict_from_row(row):
    # 실무용 간단 판정 규칙
    # 1) 방향성: 재사용군이 높아야 하는 가설은 diff>0, 미재사용군이 높아야 하는 가설은 diff<0
    # 2) 유의성: permutation p < 0.10이면 신호 있음
    p = row.get("perm_pvalue", np.nan)
    diff = row.get("diff(reuse-nonreuse)", np.nan)
    name = str(row.get("가설", ""))

    expected_positive = any(k in name for k in ["R1", "R2", "R3", "R4"])
    expected_negative = any(k in name for k in ["N1", "N2"])

    if pd.isna(diff):
        return "판단보류(데이터부족)"

    direction_ok = (expected_positive and diff > 0) or (expected_negative and diff < 0)

    if pd.isna(p):
        return "방향성만 확인"

    if direction_ok and p < 0.10:
        return "가설 지지"
    if (not direction_ok) and p < 0.10:
        return "가설 기각"
    return "약한 신호/추가데이터 필요"

summary_rows = []
for df in [reuse_hyp_df, nonreuse_hyp_df]:
    if df is None or len(df) == 0:
        continue
    tmp = df.copy()
    tmp["판정"] = tmp.apply(verdict_from_row, axis=1)
    summary_rows.append(tmp[["가설", "diff(reuse-nonreuse)", "perm_pvalue", "판정"]])

summary_df = pd.concat(summary_rows, ignore_index=True) if summary_rows else pd.DataFrame()
print_md_table("가설 검증 판정 요약", summary_df)

# 보고용 한 줄 코멘트
if not summary_df.empty:
    print("\n[임원 보고 멘트 샘플]")
    for _, r in summary_df.iterrows():
        print(f"- {r['가설']}: {r['판정']} (차이={r['diff(reuse-nonreuse)']:.4f}, p={r['perm_pvalue']:.4f})")
